# Lab 2 - CyberSleuth: an agent you can run

In the paper, CyberSleuth reads a packet capture, searches the web, and writes a forensic report.

The original needs paid LLM APIs, a Google search key and a custom framework.

Here you rebuild a toy version from parts anyone with a GitHub account can use:

| Part | In the paper | Here |
| --- | --- | --- |
| the **harness** (loop, tools, sub-agents) | custom Python | [OpenCode](https://opencode.ai), an open-source Claude Code |
| the **LLM** | GPT-4o ... GPT-5, paid | whatever **GitHub Copilot** gives your account |
| the **environment** | `tshark` | `tshark` |
| **web search** | Google Custom Search | OpenCode's `websearch` tool |
| the **incidents** | 30 | 4 of the same traces |

An architecture here is a Markdown file of about 30 lines. Read them in
[`.opencode/agents/`](.opencode/agents/): they are the whole design.

**What you do, in order**

1. Read a trace yourself, the way the agent will
2. Run one agent and read its report
3. Write the grader
4. Compare the single agent with the Tshark Expert architecture
5. Take web search away and see what breaks

Cells marked `# TODO` are yours.

*⏰ Budget: about two hours.*

> **Your Copilot quota.** Every agent run uses your Copilot allowance.
>
> This notebook makes about a dozen runs, and each one is cached in `labs/data/runs/`.
>
> Hence, re-executing a cell costs nothing. Read the rules in [`labs/README.md`](../README.md#copilot-the-rules) before you start.

In [ ]:
# Setup - checks the two tools and your Copilot login, then fetches the evidence.
import shutil, subprocess, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cybersleuth_lab.py").exists())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

import pandas as pd
import labs_lib

for tool in ("opencode", "tshark"):
    assert shutil.which(tool), f"`{tool}` is not installed - see labs/README.md"
logins = subprocess.run(["opencode", "auth", "list"], capture_output=True, text=True).stdout
assert "copilot" in logins.lower(), "not logged in: run `opencode auth login` in a terminal and pick GitHub Copilot"

MODEL = "github-copilot/gpt-5-mini"   # see labs/README.md before you change this
TRUTH = labs_lib.fetch_incidents()     # the answers - the agent never sees these
print("ready -", ROOT)
list(TRUTH)

## Part 1 - Be the agent for ten minutes

The lecture showed that a forensic agent's whole environment is a handful of
`tshark` commands. 

Before you watch an agent use them, use them yourself.

In [ ]:
def tshark(incident, *args):
    """Run tshark on one incident's trace and return what it prints."""
    pcap = labs_lib.EVIDENCE_DIR / f"{incident}.pcap"
    return subprocess.run(["tshark", "-r", str(pcap), *args], capture_output=True, text=True).stdout


print(tshark("incident-1", "-q", "-z", "conv,tcp"))

A list of connections tells you who talked to whom, but not what they said. 

Get the HTTP exchange itself, one line per packet: the request URI, the response
status code, and the `Server` header.

*Hint:* `-Y http` keeps only HTTP packets, and `-T fields -e <field>` prints the
fields you name. 

The field names are `http.request.uri`, `http.response.code` and `http.server`.

In [ ]:
def http_exchange(incident):
    """URI, status code and Server header of every HTTP packet of `incident`."""
    # TODO: one call to the `tshark` function above, using the hint's flags
    raise NotImplementedError("your turn")


print(http_exchange("incident-1"))

In [ ]:
print(http_exchange("incident-2"))

**Question 1.** From these lines alone: 
- Which service is under attack, and which version?
- What is the attacker trying to read? Did it work in incident 1, and in incident 2?

**Now name the CVE**. How sure are you, and where would you check? 

Keep your answer: the agents have to make the same call.

## Part 2 - One agent

Open [`.opencode/agents/sleuth.md`](.opencode/agents/sleuth.md). It is the single-agent (SA)
architecture, in full:

In [ ]:
print((labs_lib.SLEUTH_DIR / ".opencode/agents/sleuth.md").read_text())

The frontmatter is the tool list.
Under `permission`, `bash` is denied except for commands that start with `tshark`, and `websearch` is allowed.

The body is the system prompt.

The task is the paper's own, plus one line asking for a JSON verdict you can grade:

In [ ]:
print(labs_lib.TASK.format(pcap="evidence/incident-1.pcap"))

`investigate` runs `opencode run --agent sleuth ...` in `labs/cybersleuth/`, in the background, and parses what came back.

The first call takes a minute or two.

In [ ]:
run = labs_lib.investigate("sleuth", "incident-1", MODEL)
print(run.report)

In [ ]:
print("Stats about the run:")
print(f"{len(run.tools)} tool calls, {run.tokens:,} tokens, ${run.cost:.3f} = {run.cost * 100:.1f} Copilot AI Credits")
run.tools

### What just happened

`opencode` is the **harness**: the program that sits between you and the LLM.
The LLM never runs anything itself. At each **step** it reads the whole
conversation so far and writes one of two things: a request for a tool, or its
final answer. 

OpenCode checks the request against the `permission` block of
`sleuth.md`, runs it, and appends the output to the conversation. Then it asks
the LLM again.

In Python-like pseudo-code (do not run it):

```python
messages = [agent_prompt, task]             # the body of sleuth.md + the task above
for step in range(30):                      # `steps: 30` in the frontmatter
    reply = llm(messages, tools=allowed)    # allowed = the `permission` block
    if reply.is_final_answer:
        break                               # the report you just read
    output = harness.run(reply.tool_call)   # OpenCode runs tshark, not the LLM
    messages += [reply, output]             # the LLM only "remembers" this list
```

The LLM keeps nothing between two calls. Its only memory is `messages`, and
OpenCode sends all of it at every step.

That loop is the whole agent. `labs_lib.show_steps` prints your run as that
loop, one LLM call per step:

- `read`: the tokens the model read at that step: the prompt, the task, and every earlier step
- `wrote`: the tokens it wrote, including reasoning you never get to see
- under each call, the first lines of what `tshark` or the web search sent back

In [ ]:
labs_lib.show_steps(run)

`labs_lib.plot_context` draws the `read` column as bars, for your run and for a
`sleuth` run on incident 2.

The dark part of each bar is what the previous step already read, and the light part is what is new since then. Big jumps are labelled with the tool whose output caused them.

In [ ]:
labs_lib.plot_context(run, labs_lib.investigate("sleuth", "incident-2", MODEL))

> **Watch it live.** In a terminal:
>
> ```sh
> cd labs/cybersleuth
> OPENCODE_ENABLE_EXA=1 opencode
> ```
>
> Press <kbd>Tab</kbd> until the agent is `sleuth`, then paste the task printed
> above (the text starting with *"Analyse the PCAP file"*). You see every command
> as it runs. This is a new run: it is not cached, and it spends about 1 AI
> Credit.

----

**Question 2.** Pick two claims in the report. 

- Which tool call backs each of them? 
- When did the agent search the web: before or after it had the server version?

The lecture's Appendix B showed that this order decides the run.

----

Now the plot: each bar contains the one before it.

a. Find the tallest light part in your plot. Use `show_steps`: what did the
   step before it get back?

b. Why does the model read the dark part again at every step? Look at what
   `llm()` receives in the pseudo-code.

c. Suppose a run starts at 3,000 tokens and every step adds 300. Roughly how
   many tokens does a 30-step run read in total, and a 6-step one? What does
   that mean for the cost of long runs?

## Part 3 - The grader

CyberSleuth is graded on three checkpoints (Sec. 6.2): the **service**, the exact **CVE**, and whether the **attack succeeded**. 

In the paper an expert reads the report. You will read the JSON block instead.

`labs_lib.verdict(report)` extracts it. Write `grade`, which returns the three checkpoints as booleans.

 **N.b.**: Keep the service check lenient: the ground truth says `"apache http server"`, and the agent writes `"Apache HTTP Server 2.4.49"`.

In [ ]:
print(labs_lib.verdict(run.report))
print(TRUTH["incident-1"])

In [ ]:
def grade(verdict: dict, truth: dict) -> dict:
    # TODO: return {"service": bool, "cve": bool, "success": bool}
    raise NotImplementedError("your turn")


truth = {"service": "apache http server", "cve": "CVE-2021-41773", "success": True}
assert grade({"service": "Apache HTTP Server 2.4.49", "cve": "cve-2021-41773", "success": True}, truth) == {"service": True, "cve": True, "success": True}
assert grade({"service": "Apache 2.4.49", "cve": "CVE-2021-42013", "success": True}, truth)["cve"] is False
assert grade({"service": "nginx", "cve": "CVE-2021-41773", "success": "true"}, truth) == {"service": False, "cve": True, "success": False}
assert grade({}, truth) == {"service": False, "cve": False, "success": False}
grade(labs_lib.verdict(run.report), TRUTH["incident-1"])

**Question 3.** The third test fails a verdict whose `success` is the *string* `"true"`. Is that fair?

And the service check would accept `"Apache Tomcat"` as Apache HTTP Server. Why is a lenient grader risky when you compare two architectures?

## Part 4 - Single agent vs Tshark Expert

The Tshark Expert architecture (TEA) splits the work. 

The main agent cannot run anything: `bash` is denied, and the only sub-agent it may call is `tshark-expert`, which has `tshark` and nothing else.

> **Our `sleuth` is not the paper's SA.** In the paper, the single agent gets
> the whole packet list pasted into its prompt, and its only tool reads one
> packet at a time. Ours runs `tshark` itself, with filters. It already has the
> precise queries that TEA gets from its expert, without paying for a
> sub-agent. Keep that in mind before comparing your numbers with the paper's
> Table 2.

In [ ]:
for name in ("sleuth-tea", "tshark-expert"):
    print(f"==================== {name}.md")
    print((labs_lib.SLEUTH_DIR / ".opencode/agents" / f"{name}.md").read_text())

In `run.tools`, the expert's own commands appear right after the `task` call
that started them, as `tshark-expert:bash`. They are part of the run's cost, so
they are counted.

-----

**Question**: Run both architectures on all four incidents: eight runs, two of which you already have in the cache.

Collect one row per run, with the three checkpoints, the number of tool calls, the tokens and the cost in AI Credits.

*This cell takes a while, so start it and read on.*

In [ ]:
AGENTS = ["sleuth", "sleuth-tea"]

# TODO: for every agent and every incident, run labs_lib.investigate and grade its verdict
# TODO: keep one dict per run: agent, incident, the three checkpoints, said_cve (the
#       CVE it named), tools (how many calls), tokens, credits (run.cost * 100)
# TODO: turn the list into a DataFrame called `results`
raise NotImplementedError("your turn")
results

In [ ]:
results.groupby("agent")[["service", "cve", "success", "tools", "tokens", "credits"]].mean().round(2)

### Inside a TEA run

Below is the Tshark Expert run on incident 1, the incident of your single-agent
run in Part 2. The main agent's tool calls are now `task`: each one is a
question, in plain English, for the expert (`->`). The expert runs a loop of its
own, with its own messages, indented under the question. Only its final answer
goes back to the main agent (`<-`).

In [ ]:
labs_lib.show_steps(labs_lib.investigate("sleuth-tea", "incident-1", MODEL), lines=1)

The plot from Part 2, now for the two main agents on incident 1. Only the main
loop is drawn: the expert's own steps never enter the main agent's
conversation. A `task output` is the expert's answer.

In [ ]:
labs_lib.plot_context(labs_lib.investigate("sleuth", "incident-1", MODEL),
                      labs_lib.investigate("sleuth-tea", "incident-1", MODEL))

----
**Question 4.** Incident 2 runs Apache **2.4.50**. Two facts you need:

- **CVE-2021-41773** hit Apache 2.4.49. Its exploit hides `../` by encoding one
  dot: `.%2e/`. Apache 2.4.50 was released to block exactly that.
- **CVE-2021-42013** is the bypass of that fix, and it hits 2.4.50. Its exploit
  encodes the dot *twice*: `%2e` becomes `%%32%65`, so the request contains
  `.%%32%65/`.

The benchmark labels incident 2 as **CVE-2021-42013**.

a. Compare the URIs of incidents 1 and 2 (Part 1). Which of the two exploits
   does the attacker send in incident 2?
b. Given your answer to (a), what does the `400 Bad Request` tell you?
c. Both agents answered CVE-2021-42013 and got full marks. Which line of the
   trace supports that CVE: a request, or the `Server` header? Is the report's
   claim that the requests "match CVE-2021-42013" true? What does this say
   about grading only the final label?

----
**Question 5.** Read the `tools` and `credits` columns. What does the second
agent buy you, and what does it cost? Would you pay that on 10 000 incidents?

Then read the TEA run above. Which `tshark` output does the main agent never
see, and what does it read instead? The plot shows what each main agent reads
at each step: is TEA's context really smaller? What makes its bars grow?

----

**Question 6.** Answering "the attack succeeded" every time already scores 2/4
here. With four incidents and one run each, can you tell which architecture is
better? What would you need? (The paper ran every incident three times.)

## Part 5 - Take the web away

Vite's CVE-2025-30208 was disclosed in March 2025. Did the agents find it by
searching, or did they already know it? Count the searches:

In [ ]:
for agent in AGENTS:
    for incident in ("incident-3", "incident-4"):
        run = labs_lib.investigate(agent, incident, MODEL)
        print(f"{agent:<11} {incident}  websearch x{run.tools.count('websearch')}  ->  {labs_lib.verdict(run.report).get('cve')}")

Changing the architecture means changing a file. Create
`.opencode/agents/sleuth-offline.md`: the same as `sleuth.md`, but with
`websearch` and `webfetch` denied. Then run it on the Vite pair.

In [ ]:
agents_dir = labs_lib.SLEUTH_DIR / ".opencode/agents"
# TODO: write sleuth-offline.md from sleuth.md, with the two web tools denied
raise NotImplementedError("your turn")
assert "websearch: deny" in (agents_dir / "sleuth-offline.md").read_text()

In [ ]:
for incident in ("incident-3", "incident-4"):
    run = labs_lib.investigate("sleuth-offline", incident, MODEL)
    found = labs_lib.verdict(run.report)
    print(incident, found, grade(found, TRUTH[incident]))

**Question 7.** Without the web, what CVE does the agent give for Vite? 
Is it still right? Does the report admit it is guessing? 

If a run has no verdict at all, print its `report`: what did the agent do instead?

`sleuth-offline.md` still tells the agent to use `websearch` in the body, which it no longer has. Is the agent's behaviour a bug in the agent, or in your architecture?